# 02 — Attention Extraction

Для каждого baseline-промпта прогоняем CogVideoX через `temporal_hook_context`,
собираем temporal distance профили `{layer_idx: Tensor(heads, 2T-1)}`,
сохраняем в `results/profiles/` и визуализируем heatmap + per-layer кривые.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import yaml
from diffusers import CogVideoXPipeline
from omegaconf import OmegaConf

sys.path.insert(0, str(Path("..").resolve()))
from src.hooks import temporal_hook_context

In [ ]:
# Latent grid dims: 49 frames @ 480×720
# After 3D VAE (4× temporal, 8× spatial): T=13, H=60, W=90
# After CogVideoX patch_size=2 spatial patchification: H=30, W=45
T, H, W = 13, 30, 45
NUM_LAYERS = 42
RESULTS_DIR = Path("../results/profiles")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

pipe = CogVideoXPipeline.from_pretrained(
    "/workspace/hf_cache/CogVideoX-5b",
    torch_dtype=torch.bfloat16,
    use_safetensors=True,
).to("cuda")

In [ ]:
cfg = OmegaConf.load("../configs/baseline_prompts.yaml")
gen = cfg.generation
prompts = cfg.prompts
print(f"{len(prompts)} prompts loaded")

## Извлечение профилей

Для каждого промпта: запускаем пайплайн внутри `temporal_hook_context`, профили усредняются
по всем шагам денойзинга автоматически через `HookState._counts`.

In [ ]:
all_profiles: dict[str, dict[int, torch.Tensor]] = {}

with torch.inference_mode():
    for prompt_cfg in prompts:
        pid = prompt_cfg.id
        out_path = RESULTS_DIR / f"{pid}.pt"

        if out_path.exists():
            print(f"[skip] {pid} — already extracted")
            all_profiles[pid] = torch.load(out_path, map_location="cpu")
            continue

        print(f"[run ] {pid} ({prompt_cfg.motion_profile})")
        with temporal_hook_context(pipe.transformer, T=T, H=H, W=W) as state:
            pipe(
                prompt=prompt_cfg.text,
                negative_prompt=gen.negative_prompt,
                num_frames=gen.num_frames,
                num_inference_steps=gen.num_inference_steps,
                guidance_scale=gen.guidance_scale,
            )

        profiles_cpu = {k: v.cpu() for k, v in state.profiles.items()}
        torch.save(profiles_cpu, out_path)
        all_profiles[pid] = profiles_cpu
        print(f"       saved -> {out_path}  ({len(profiles_cpu)} layers)")

print("\nDone.")

## Визуализация: heatmap layer × head

Для каждого промпта: доминирующая temporal distance (argmax по оси `2T-1`) для каждой головы каждого слоя.
Значение `d=0` соответствует индексу `T-1=12` в профиле (self-attention, distance=0).

In [ ]:
def dominant_distance_heatmap(profiles: dict[int, torch.Tensor], T: int) -> np.ndarray:
    """profiles: {layer_idx: (heads, 2T-1)}  →  ndarray(num_layers, heads)"""
    layers = sorted(profiles)
    num_heads = next(iter(profiles.values())).shape[0]
    heatmap = np.zeros((len(layers), num_heads), dtype=np.float32)
    for row, layer_idx in enumerate(layers):
        # shift to signed temporal distance: [-T+1, T-1]
        dominant_idx = profiles[layer_idx].argmax(dim=-1).numpy()  # (heads,)
        heatmap[row] = dominant_idx - (T - 1)
    return heatmap

In [ ]:
motion_groups = {
    "global_motion": ["drone_city", "highway_drive", "coast_pan"],
    "object_motion": ["walk_park", "dance_spotlight", "dog_fetch"],
    "fluid_periodic": ["ocean_waves", "flag_wind", "campfire_smoke"],
    "slow_discrete": ["leaves_falling", "snowfall_street", "candle_flame"],
    "near_static": ["still_lake"],
}

# One representative per group for the main figure
representatives = ["drone_city", "dance_spotlight", "ocean_waves", "leaves_falling", "still_lake"]

In [ ]:
n = len(representatives)
fig, axes = plt.subplots(1, n, figsize=(4 * n, 8), sharey=True)

vmin, vmax = -(T - 1), T - 1

for ax, pid in zip(axes, representatives):
    if pid not in all_profiles:
        ax.set_title(f"{pid}\n(missing)")
        continue
    hm = dominant_distance_heatmap(all_profiles[pid], T)
    im = ax.imshow(hm, aspect="auto", cmap="RdBu_r", vmin=vmin, vmax=vmax, origin="upper")
    ax.set_title(pid.replace("_", "\n"), fontsize=9)
    ax.set_xlabel("Head")

axes[0].set_ylabel("Layer")
fig.colorbar(im, ax=axes, orientation="vertical", fraction=0.02, label="Dominant temporal distance (frames)")
fig.suptitle("Dominant temporal distance — layer × head", fontsize=12)
plt.savefig(RESULTS_DIR / "heatmap_dominant_distance.png", dpi=150, bbox_inches="tight")
plt.show()

## Визуализация: temporal distance профили по слоям

Для одного промпта показываем средний профиль (по головам) для выбранных слоёв.
Кривая показывает, насколько внимание сконцентрировано на соседних vs. далёких кадрах.

In [ ]:
PROBE_LAYERS = [0, 5, 10, 15, 20, 25, 30, 35, 41]
PROBE_PROMPT = "ocean_waves"  # fluid_periodic — наглядный паттерн

dist_axis = np.arange(-(T - 1), T)  # [-12, ..., 12]

fig, axes = plt.subplots(3, 3, figsize=(12, 9), sharex=True)
axes_flat = axes.flatten()

profiles = all_profiles.get(PROBE_PROMPT, {})
for ax, layer_idx in zip(axes_flat, PROBE_LAYERS):
    if layer_idx not in profiles:
        ax.set_title(f"layer {layer_idx} (missing)")
        continue
    p = profiles[layer_idx].float().numpy()  # (heads, 2T-1)
    mean_p = p.mean(axis=0)
    ax.bar(dist_axis, mean_p, color="steelblue", alpha=0.8, width=0.8)
    ax.axvline(0, color="red", linewidth=0.8, linestyle="--")
    ax.set_title(f"layer {layer_idx}", fontsize=9)
    ax.set_xlim(-(T - 1) - 0.5, (T - 1) + 0.5)

for ax in axes_flat:
    ax.set_xlabel("Temporal distance (frames)")

fig.suptitle(f"Temporal distance profiles — {PROBE_PROMPT}", fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"profiles_{PROBE_PROMPT}.png", dpi=150, bbox_inches="tight")
plt.show()

## Сравнение motion profiles: средний профиль по группам

Усредняем профили всех промптов одной группы (по `motion_profile`) и сравниваем.
Ожидаем: `global_motion` → длинные хвосты, `near_static` → концентрация на `d=0`.

In [ ]:
COMPARE_LAYER = 20  # середина сети

fig, ax = plt.subplots(figsize=(10, 4))

for group, pids in motion_groups.items():
    group_profiles = []
    for pid in pids:
        if pid in all_profiles and COMPARE_LAYER in all_profiles[pid]:
            p = all_profiles[pid][COMPARE_LAYER].float().mean(dim=0).numpy()  # (2T-1,)
            group_profiles.append(p)
    if not group_profiles:
        continue
    mean_group = np.stack(group_profiles).mean(axis=0)
    ax.plot(dist_axis, mean_group, label=group, linewidth=1.5)

ax.axvline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_xlabel("Temporal distance (frames)")
ax.set_ylabel("Mean attention weight")
ax.set_title(f"Layer {COMPARE_LAYER}: mean profile per motion group")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR / f"group_comparison_layer{COMPARE_LAYER}.png", dpi=150, bbox_inches="tight")
plt.show()